# HQNN-Parallel: ideal, clássico e ruído IBM ao longo do treinamento

Compararemos três casos em MNIST:

1. **HQNN ideal**: circuito quântico simulado sem ruído.
2. **Baseline clássico**: mesma CNN e mesmo gargalo de 20 características, substituindo os PQCs por `Linear(20,20)`.
3. **HQNN + ruído IBM**: mesma inicialização do HQNN ideal, mas com resíduos medidos em um QPU real.

Os modelos quânticos serão treinados por **8 épocas**. O HQNN ideal será salvo em todas as épocas e o QPU será calibrado nos checkpoints \(0,2,4,6,8\). Assim, o ruído usado no treinamento ruidoso pode mudar conforme os pesos quânticos evoluem.

## 1. Bibliotecas e configuração

In [ ]:
import json
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import pennylane as qml

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda")
print(torch.cuda.get_device_name(0))

In [ ]:
BATCH_SIZE = 32
EPOCHS = 8
LR = 1e-3
N_TRAIN = 2000
N_TEST = 500

N_QUBITS = 5
N_LAYERS = 3
N_CIRCUITS = 4
N_QFEATURES = N_QUBITS * N_CIRCUITS

CALIB_EPOCHS = [0, 2, 4, 6, 8]
CALIB_INDICES = [0, 1, 2]
N_REPEATS = 2
IBM_SHOTS = 1000

OUTDIR = Path("hqnn_runs")
OUTDIR.mkdir(exist_ok=True)

## 2. MNIST

A configuração curta usa 2000 imagens de treino e 500 de teste. Para o conjunto completo, use `N_TRAIN = 60000` e `N_TEST = 10000`.

In [ ]:
transform = transforms.ToTensor()

train_full = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_full = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_dataset = Subset(train_full, range(N_TRAIN))
test_dataset = Subset(test_full, range(N_TEST))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Treino:", len(train_dataset))
print("Teste:", len(test_dataset))

## 3. Circuito quântico

A CNN produz \(32\times7\times7=1568\) características. A primeira camada densa reduz isso para 20 valores, divididos entre quatro PQCs de 5 qubits.

In [ ]:
qdev = qml.device("lightning.gpu", wires=N_QUBITS, shots=None)

@qml.qnode(qdev, interface="torch", diff_method="adjoint")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="X")
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliY(i)) for i in range(N_QUBITS)]

weight_shapes = {"weights": (N_LAYERS, N_QUBITS, 3)}

## 4. Modelos

In [ ]:
class ClassicalCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return torch.flatten(self.conv(x), 1)


class ParallelQuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.quantum_layers = nn.ModuleList([
            qml.qnn.TorchLayer(quantum_circuit, weight_shapes)
            for _ in range(N_CIRCUITS)
        ])

    def forward(self, x):
        return torch.cat([
            layer(x[:, i*N_QUBITS:(i+1)*N_QUBITS])
            for i, layer in enumerate(self.quantum_layers)
        ], dim=1)


class HQNNParallel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = ClassicalCNN()
        self.fc1 = nn.Linear(1568, N_QFEATURES)
        self.bn1 = nn.BatchNorm1d(N_QFEATURES)
        self.quantum = ParallelQuantumLayer()
        self.fc2 = nn.Linear(N_QFEATURES, 10)

    def features20(self, x):
        return F.relu(self.bn1(self.fc1(self.cnn(x))))

    def forward(self, x, qnoise=None):
        x = self.quantum(self.features20(x))
        if qnoise is not None:
            x = x + qnoise
        return self.fc2(x)


class ClassicalBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = ClassicalCNN()
        self.fc1 = nn.Linear(1568, N_QFEATURES)
        self.bn1 = nn.BatchNorm1d(N_QFEATURES)
        self.hidden = nn.Linear(N_QFEATURES, N_QFEATURES)
        self.fc2 = nn.Linear(N_QFEATURES, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(self.cnn(x))))
        x = F.relu(self.hidden(x))
        return self.fc2(x)


def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
base_quantum = HQNNParallel().to(device)
initial_quantum_state = copy.deepcopy(base_quantum.state_dict())

base_classical = ClassicalBaseline().to(device)

print("Parâmetros treináveis HQNN:", n_params(base_quantum))
print("Parâmetros treináveis clássico:", n_params(base_classical))
print("Parâmetros quânticos:", sum(
    p.numel()
    for name, p in base_quantum.named_parameters()
    if "quantum" in name and p.requires_grad
))

## 5. Salvamento e carregamento

In [ ]:
def salvar_checkpoint(model, optimizer, history, epoch, tag):
    payload = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "history": history
    }

    torch.save(payload, OUTDIR / f"{tag}_epoch_{epoch:03d}.pt")
    torch.save(payload, OUTDIR / f"{tag}_latest.pt")

    with open(OUTDIR / f"{tag}_history.json", "w") as f:
        json.dump(history, f, indent=2)


def carregar_checkpoint(tag, epoch=None, model=None, lr=LR):
    path = OUTDIR / (f"{tag}_latest.pt" if epoch is None else f"{tag}_epoch_{epoch:03d}.pt")

    if model is None:
        model = HQNNParallel().to(device)

    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    if checkpoint["optimizer_state"] is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state"])

    return model, optimizer, checkpoint["history"], checkpoint["epoch"]

## 6. Avaliação e treinamento

In [ ]:
@torch.no_grad()
def avaliar(model, loader):
    model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)

        loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.numel()

    return loss_sum / total, correct / total


def escolher_ruido(epoch, noise_by_epoch):
    available = np.array(sorted(noise_by_epoch))
    nearest = int(available[np.argmin(np.abs(available - epoch))])
    return noise_by_epoch[nearest], nearest


def treinar(model, tag, epochs=EPOCHS, noise_by_epoch=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    salvar_checkpoint(model, optimizer, history, 0, tag)

    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            if noise_by_epoch is None:
                logits = model(images)
                calib_epoch = None
            else:
                pool, calib_epoch = escolher_ruido(epoch, noise_by_epoch)
                idx = np.random.randint(0, len(pool), size=len(images))
                qnoise = torch.as_tensor(pool[idx], dtype=torch.float32, device=device)
                logits = model(images, qnoise=qnoise)

            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * labels.numel()
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.numel()

        train_loss = loss_sum / total
        train_acc = correct / total
        test_loss, test_acc = avaliar(model, test_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        salvar_checkpoint(model, optimizer, history, epoch, tag)

        extra = "" if calib_epoch is None else f" | calibração IBM={calib_epoch}"

        print(
            f"Época {epoch:02d}/{epochs} | "
            f"train loss={train_loss:.4f} acc={train_acc:.4f} | "
            f"test loss={test_loss:.4f} acc={test_acc:.4f}"
            f"{extra}"
        )

    return model, history

## 7. HQNN ideal — 8 épocas

In [ ]:
model_ideal = HQNNParallel().to(device)
model_ideal.load_state_dict(copy.deepcopy(initial_quantum_state))

model_ideal, history_ideal = treinar(
    model_ideal,
    tag="ideal",
    epochs=EPOCHS
)

## 8. Baseline clássico — 8 épocas

In [ ]:
model_classical = ClassicalBaseline().to(device)

model_classical, history_classical = treinar(
    model_classical,
    tag="classical",
    epochs=EPOCHS
)

## 9. Convergência HQNN ideal × clássico

In [ ]:
epochs_axis = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs_axis, history_ideal["test_loss"], marker="o", label="HQNN ideal")
plt.plot(epochs_axis, history_classical["test_loss"], marker="o", label="Clássico")
plt.xlabel("Época")
plt.ylabel("Test loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs_axis, history_ideal["test_acc"], marker="o", label="HQNN ideal")
plt.plot(epochs_axis, history_classical["test_acc"], marker="o", label="Clássico")
plt.xlabel("Época")
plt.ylabel("Test accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. IBM Quantum

A calibração usa os checkpoints \(0,2,4,6,8\), sempre com as mesmas imagens. Isso permite medir se o erro do QPU muda conforme os pesos aprendidos mudam.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator

service = QiskitRuntimeService()

backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=N_QUBITS
)

print("Backend:", backend.name)

In [ ]:
def make_qiskit_circuit(features5, weights):
    qc = QuantumCircuit(N_QUBITS)

    for q in range(N_QUBITS):
        qc.rx(float(features5[q]), q)

    for layer in range(weights.shape[0]):
        for q in range(N_QUBITS):
            phi, theta, omega = weights[layer, q]
            qc.rz(float(phi), q)
            qc.ry(float(theta), q)
            qc.rz(float(omega), q)

        r = layer + 1

        for q in range(N_QUBITS):
            qc.cx(q, (q + r) % N_QUBITS)

    return qc


Y_observables = [
    SparsePauliOp("IIIIY"),
    SparsePauliOp("IIIYI"),
    SparsePauliOp("IIYII"),
    SparsePauliOp("IYIII"),
    SparsePauliOp("YIIII")
]

pm = generate_preset_pass_manager(backend=backend, optimization_level=2)
estimator = Estimator(mode=backend)

## 11. Validação PennyLane × Qiskit

In [ ]:
model_check, _, _, _ = carregar_checkpoint("ideal", epoch=0)

image_check = test_dataset[0][0].unsqueeze(0).to(device)

with torch.no_grad():
    features_t = model_check.features20(image_check)
    pl5 = model_check.quantum.quantum_layers[0](features_t[:, :5]).detach().cpu().numpy()[0]

features5 = features_t[0, :5].detach().cpu().numpy()
weights5 = model_check.quantum.quantum_layers[0].weights.detach().cpu().numpy()

qc = make_qiskit_circuit(features5, weights5)
state = Statevector.from_instruction(qc)
qiskit5 = np.array([np.real(state.expectation_value(obs)) for obs in Y_observables])

print("Máxima diferença:", np.max(np.abs(pl5 - qiskit5)))

In [ ]:
def quantum_weights_np(model):
    return [
        layer.weights.detach().cpu().numpy()
        for layer in model.quantum.quantum_layers
    ]


def run_quantum_layer_ibm(features20, weights4, shots=IBM_SHOTS):
    pubs = []

    for i in range(N_CIRCUITS):
        f5 = features20[i*N_QUBITS:(i+1)*N_QUBITS]
        qc = make_qiskit_circuit(f5, weights4[i])
        isa_qc = pm.run(qc)
        isa_obs = [obs.apply_layout(isa_qc.layout) for obs in Y_observables]
        pubs.append((isa_qc, isa_obs))

    estimator.options.default_shots = shots
    job = estimator.run(pubs)

    print("Job:", job.job_id())

    result = job.result()

    means = np.concatenate([
        np.asarray(r.data.evs, dtype=float)
        for r in result
    ])

    stds = np.concatenate([
        np.asarray(r.data.stds, dtype=float)
        for r in result
    ])

    return means, stds

## 12. Calibração do ruído nos checkpoints

Em cada checkpoint calculamos:

\[
\Delta q=q_{\rm IBM}-q_{\rm ideal}.
\]

Também calculamos MAE, bias e correlação. Cada calibração é salva separadamente.

In [ ]:
noise_by_epoch = {}
noise_metrics = []

for epoch in CALIB_EPOCHS:
    print(f"\nCalibração da época {epoch}")

    model_epoch, _, _, _ = carregar_checkpoint("ideal", epoch=epoch)
    model_epoch.eval()

    weights4 = quantum_weights_np(model_epoch)

    residuals_epoch = []
    shot_stds_epoch = []
    ideal_epoch = []
    ibm_epoch = []

    for idx in CALIB_INDICES:
        image = test_dataset[idx][0].unsqueeze(0).to(device)

        with torch.no_grad():
            features_t = model_epoch.features20(image)
            ideal = model_epoch.quantum(features_t).detach().cpu().numpy()[0]

        features_np = features_t[0].detach().cpu().numpy()

        for rep in range(N_REPEATS):
            measured, std = run_quantum_layer_ibm(
                features_np,
                weights4,
                shots=IBM_SHOTS
            )

            residuals_epoch.append(measured - ideal)
            shot_stds_epoch.append(std)
            ideal_epoch.append(ideal)
            ibm_epoch.append(measured)

    residuals_epoch = np.asarray(residuals_epoch)
    shot_stds_epoch = np.asarray(shot_stds_epoch)
    ideal_epoch = np.asarray(ideal_epoch)
    ibm_epoch = np.asarray(ibm_epoch)

    noise_by_epoch[epoch] = residuals_epoch

    flat_ideal = ideal_epoch.ravel()
    flat_ibm = ibm_epoch.ravel()

    mae = np.mean(np.abs(flat_ibm - flat_ideal))
    bias = np.mean(flat_ibm - flat_ideal)
    corr = np.corrcoef(flat_ideal, flat_ibm)[0, 1]

    noise_metrics.append({
        "epoch": epoch,
        "mae": float(mae),
        "bias": float(bias),
        "correlation": float(corr)
    })

    np.savez(
        OUTDIR / f"ibm_noise_epoch_{epoch:03d}.npz",
        residuals=residuals_epoch,
        shot_stds=shot_stds_epoch,
        ideal=ideal_epoch,
        ibm=ibm_epoch,
        shots=IBM_SHOTS
    )

    print(f"MAE={mae:.4f} | bias={bias:+.4f} | r={corr:.4f}")

with open(OUTDIR / "ibm_noise_metrics.json", "w") as f:
    json.dump(noise_metrics, f, indent=2)

## 13. Carregar calibrações salvas

In [ ]:
noise_by_epoch = {}

for epoch in CALIB_EPOCHS:
    d = np.load(OUTDIR / f"ibm_noise_epoch_{epoch:03d}.npz")
    noise_by_epoch[epoch] = d["residuals"]

with open(OUTDIR / "ibm_noise_metrics.json") as f:
    noise_metrics = json.load(f)

print({k: v.shape for k, v in noise_by_epoch.items()})

## 14. Evolução do ruído

O MAE mede o tamanho médio do desvio. O bias mede um deslocamento sistemático. A correlação mostra quanto o QPU preserva a estrutura das saídas ideais.

In [ ]:
metric_epochs = np.array([m["epoch"] for m in noise_metrics])
mae = np.array([m["mae"] for m in noise_metrics])
bias = np.array([m["bias"] for m in noise_metrics])
corr = np.array([m["correlation"] for m in noise_metrics])

plt.figure(figsize=(7, 4))
plt.plot(metric_epochs, mae, marker="o", label="MAE")
plt.plot(metric_epochs, np.abs(bias), marker="o", label="|bias|")
plt.xlabel("Época")
plt.ylabel("Erro em <Y>")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(metric_epochs, corr, marker="o")
plt.xlabel("Época")
plt.ylabel("Correlação ideal × IBM")
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)
plt.show()

## 15. Ideal × IBM no checkpoint final

In [ ]:
last = np.load(OUTDIR / f"ibm_noise_epoch_{CALIB_EPOCHS[-1]:03d}.npz")

xideal = last["ideal"].ravel()
xibm = last["ibm"].ravel()
xstd = last["shot_stds"].ravel()

plt.figure(figsize=(6, 6))
plt.errorbar(xideal, xibm, yerr=xstd, fmt="o", capsize=2, alpha=0.7)
plt.plot([-1, 1], [-1, 1], "--")
plt.xlim(-1, 1)
plt.ylim(-1, 1)
plt.xlabel("Simulador ideal")
plt.ylabel("IBM QPU")
plt.title(f"Época {CALIB_EPOCHS[-1]}")
plt.grid(alpha=0.3)
plt.show()

## 16. HQNN com ruído IBM — 8 épocas

O modelo ruidoso parte exatamente dos mesmos pesos iniciais do HQNN ideal. Para cada época é usada a calibração IBM mais próxima:

\[
q_{\rm ruidoso}=q_{\rm ideal}+\Delta q_{\rm IBM}.
\]

In [ ]:
model_noisy = HQNNParallel().to(device)
model_noisy.load_state_dict(copy.deepcopy(initial_quantum_state))

model_noisy, history_noisy = treinar(
    model_noisy,
    tag="ibm_noise",
    epochs=EPOCHS,
    noise_by_epoch=noise_by_epoch
)

## 17. Comparação final da convergência

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(epochs_axis, history_classical["test_loss"], marker="o", label="Clássico")
plt.plot(epochs_axis, history_ideal["test_loss"], marker="o", label="HQNN ideal")
plt.plot(epochs_axis, history_noisy["test_loss"], marker="o", label="HQNN + ruído IBM")
plt.xlabel("Época")
plt.ylabel("Test loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs_axis, history_classical["test_acc"], marker="o", label="Clássico")
plt.plot(epochs_axis, history_ideal["test_acc"], marker="o", label="HQNN ideal")
plt.plot(epochs_axis, history_noisy["test_acc"], marker="o", label="HQNN + ruído IBM")
plt.xlabel("Época")
plt.ylabel("Test accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print("Parâmetros treináveis")
print("Clássico:", n_params(model_classical))
print("HQNN ideal:", n_params(model_ideal))
print("HQNN + ruído IBM:", n_params(model_noisy))

print("\nResultado final")
print(f"Clássico       | loss={history_classical['test_loss'][-1]:.4f} | acc={history_classical['test_acc'][-1]:.4f}")
print(f"HQNN ideal     | loss={history_ideal['test_loss'][-1]:.4f} | acc={history_ideal['test_acc'][-1]:.4f}")
print(f"HQNN ruído IBM | loss={history_noisy['test_loss'][-1]:.4f} | acc={history_noisy['test_acc'][-1]:.4f}")

## 18. Propagação da incerteza estatística

Além do residual empírico usado no treinamento, o Estimator fornece a incerteza de cada \(\langle Y\rangle\). Aqui propagamos essa incerteza até as probabilidades de classe e a loss usando Monte Carlo.

In [ ]:
epoch = CALIB_EPOCHS[-1]
d = np.load(OUTDIR / f"ibm_noise_epoch_{epoch:03d}.npz")

q_mean = d["ibm"][0]
q_std = d["shot_stds"][0]

model_epoch, _, _, _ = carregar_checkpoint("ideal", epoch=epoch)

label = int(test_dataset[CALIB_INDICES[0]][1])
N_MC = 10000

mu = torch.as_tensor(q_mean, dtype=torch.float32, device=device)
sigma = torch.as_tensor(q_std, dtype=torch.float32, device=device)

samples = mu[None, :] + torch.randn(N_MC, N_QFEATURES, device=device) * sigma[None, :]

with torch.no_grad():
    logits = model_epoch.fc2(samples)
    probs = torch.softmax(logits, 1)

targets = torch.full((N_MC,), label, dtype=torch.long, device=device)
losses = F.cross_entropy(logits, targets, reduction="none")

pmean = probs.mean(0).cpu().numpy()
pstd = probs.std(0).cpu().numpy()

for c in range(10):
    print(f"Classe {c}: {pmean[c]:.4f} ± {pstd[c]:.4f}")

print(f"Loss: {losses.mean().item():.4f} ± {losses.std().item():.4f}")

## 19. Carregar os modelos salvos

In [ ]:
model_ideal_loaded, _, history_ideal_loaded, epoch_ideal = carregar_checkpoint("ideal")
model_noisy_loaded, _, history_noisy_loaded, epoch_noisy = carregar_checkpoint("ibm_noise")

classical_loaded = ClassicalBaseline().to(device)
classical_loaded, _, history_classical_loaded, epoch_classical = carregar_checkpoint(
    "classical",
    model=classical_loaded
)

print("Ideal:", epoch_ideal)
print("Ruído IBM:", epoch_noisy)
print("Clássico:", epoch_classical)

## Interpretação

O experimento compara uma rede clássica, um HQNN ideal e um HQNN treinado com resíduos medidos no IBM QPU.

A calibração não é considerada constante: ela é repetida em checkpoints diferentes do treinamento ideal. Dessa forma podemos verificar diretamente se MAE, bias e correlação mudam conforme os pesos quânticos mudam.

O treinamento ruidoso não executa cada batch no QPU. Ele reutiliza, por bootstrap, resíduos reais medidos no checkpoint mais próximo. Isso torna possível estudar várias épocas sem milhares de execuções físicas.

A propagação Monte Carlo é complementar: ela mostra como a incerteza estatística das medições do QPU chega até as probabilidades de classe e a cross-entropy.